In [1]:
from __future__ import annotations
import operator
import os
import re
from datetime import date , timedelta
from pathlib import Path
from typing import TypedDict , List , Optional , Literal , Annotated

from pydantic import BaseModel , Field
from langgraph.graph import StateGraph , START , END
from langgraph.types import Send

from langchain_openai import ChatOpenAI
from langchain_core.messages import SystemMessage , HumanMessage
from langchain_community.tools.tavily_search import TavilySearchResults

c:\LangGraph\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\M.LAPTOP\AppData\Local\Temp\ipykernel_10792\2047314187.py:15: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools.tavily_search import TavilySearchResults


In [2]:
class Task(BaseModel):
    id:int
    title:str

    goal:str = Field(
        ...,
        description="One sentence describing what the reader should be able to do/understand after this section",
    )
    bullets:List[str] = Field(
        ...,
        min_length=3,
        max_length=6,
        description="3-6 concrete , non-overlapping subpoints to cover in this section." ,
    )
    target_words : int = Field(
        ...,
        description="Target word count for this section(120-550)"
    )
    tags: List[str] = Field(default_factory=list)
    requires_research: bool = False
    requires_citation : bool = False
    requires_code : bool = False

class Plan(BaseModel):
    blog_title:str
    audience:str
    tone:str
    blog_kind:Literal["explainer", "tutorial", "news_roundup", "comparison", "system_design"] = "explainer"
    constrains:List[str] = Field(default_factory=list)
    tasks:List[Task]

class EvidenceItem(BaseModel):
    title:str
    url:str
    published_at : Optional[str] = None
    snippet : Optional[str] = None
    source : Optional[str] = None

class RouterDecision(BaseModel):
    needs_research:bool
    mode : Literal["closed_book" , "hybrid" , "open_book"]
    queries : List[str] = Field(default_factory=list)

class EvidencePack(BaseModel):
    evidence : List[EvidenceItem] = Field(default_factory=list)

class ImageSpec(BaseModel):
    placeholder : str = Field(
        ...,
        description="e.g.[[IMAGE_1]]")
    filename: str= Field(
            ...,
            description="Save uner images/ , e.g. qkv_flow.png"
        )  
    alt:str
    caption:str
    prompt:str = Field(
        ...,
        description="Prompt to send to the image model.")
    size: Literal["1024x1024", "1024x1536", "1536x1024"] = "1024x1024"
    quality: Literal["low" , "medium" , "high"] = "medium"
class GlobalImagePlan(BaseModel):
    md_with_placeholder:str
    images:List[ImageSpec] = Field(default_factory=list)

In [3]:
class State(TypedDict):
    topic:str
    mode:str
    needs_research: bool
    queries: Optional[Plan]

    sections:Annotated[List[tuple[int , str]] , operator.add]

    merged_md : str
    md_with_placeholders: str
    image_specs:List[dict]

    final:str

llm = ChatOpenAI(model = "gpt-4.1-mini")

In [4]:
ROUTER_SYSTEM =  """You are a routing module for a technical blog planner.

Decide whether web research is needed BEFORE planning.

Modes:
- closed_book (needs_research=false):
  Evergreen topics where correctness does not depend on recent facts (concepts, fundamentals).
- hybrid (needs_research=true):
  Mostly evergreen but needs up-to-date examples/tools/models to be useful.
- open_book (needs_research=true):
  Mostly volatile: weekly roundups, "this week", "latest", rankings, pricing, policy/regulation.

If needs_research=true:
- Output 3–10 high-signal queries.
- Queries should be scoped and specific (avoid generic queries like just "AI" or "LLM").
- If user asked for "last week/this week/latest", reflect that constraint IN THE QUERIES.
"""
def router_node(State:State) -> dict:
    topic = state["topic"]
    decider = llm.with_structured_output(RouterDecision)
    decision = decider.invoke([
        SystemMessage(content = ROUTER_SYSTEM),
        HumanMessage(content=f"Topic:{topic}"),
    ]
    )
    return {
        "needs_research": decision.needs_research,
        "mode":decision.mode,
        "queries":decision.queries,
    }
def route_next(state:State) -> str:
    return "research" if state["needs_research"] else "orchestrator"


In [ ]:
def _tavily_search(query:str , max_results:int = 5) -> List[dict]:
    tool = TavilySearchResults(max_results=max_results)
    results = tool.invoke({"query": query})

    normalized : List[dict]  = []
    for r in results or []:
        normazlized.append(
            {
                "title": r.get("title") or "",
                "url" : r.get("url") or "",
                "snippet" : r.get("published_date") or r.get("published_at") , 
                "source" : r.get("source")
            }
        )
        return normalized
RESEARCH_SYSTEM = """Given raw web search results, produce a deduplicated list of EvidenceItem objects.

Rules:
- Only include items with a non-empty url.
- Prefer relevant + authoritative sources (company blogs, docs, reputable outlets).
- If a published date is explicitly present in the result payload, keep it as YYYY-MM-DD.
  If missing or unclear, set published_at=null. Do NOT guess.
- Keep snippets short.
- Deduplicate by URL.
"""
def research_node(state:State) -> dict:
    queries = (state.get("queries" , []or []))
    max_results = 6

    raw_results : List[dict]  = []
    for q in queries:
        raw_results.extend(_tavily_search(q,max_results=max_results))
        if not raw_results:
            return{"evidence":[]}
        extractor = llm.with_structured_output(EvidencePack)
        pack = extractor.invoke(
            SystemMessage(content = RESEARCH_SYSTEM),
            HumanMessage(content = f"Raw RESULTS:\n{raw_results}"),
        )
    dedup = {}
    for e in pack.evidence:
        if e.url:
            dedup[e.url] = e
    return{"evidence": list(dedup.values())}
    print()

In [ ]:
def research_node(state:State) -> dict:
    queries = (state.get("queries" , []or []))
    max_results = 6

    raw_results : List[dict]  = []
    for q in queries:
        raw_results.extend(_tavily_search(q,max_results=max_results))
        if not raw_results:
            return{"evidence":[]}
        extractor = llm.with_structured_output(EvidencePack)
        pack = extractor.invoke(
            SystemMessage(content = RESEARCH_SYSTEM),
            HumanMessage(content = f"Raw RESULTS:\n{raw_results}"),
        )
    dedup = {}
    for e in pack.evidence:
        if e.url:
            dedup[e.url] = e
    return{"evidence": list(dedup.values())}
    print()